# Buidling a Deep Research Agent

One of the classic cross-business Agentic use cases. A Deep Research Agent is an autonomous AI system designed to conduct exhaustive, multi-step investigations into complex, open-ended questions.

Rather than instantly answering a question based on its training data or running a single search query (like traditional chatbots or basic Retrieval-Augmented Generation / RAG), a Deep Research Agent acts like a human analyst. It creates an execution plan, breaks the topic into sub-investigations, dynamically searches external databases and web sources, evaluates evidence, and synthesizes findings into a comprehensive, cited report.

### We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [2]:
load_dotenv(override=True)

True

In [3]:
# Constants 
MODEL_NAME = "gpt-5.4-mini"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

## 1. The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem.

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  
`FileSearchTool` allows retrieving information from your OpenAI Vector Stores.  
`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  
`HostedMCPTool` exposes a remote MCP server's tools to the model.  
`ImageGenerationTool` generates images from a prompt.  
`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand.  

#### Important note - API charge of WebSearchTool
This currently costs 1 cent per call for OpenAI WebSearchTool. We'll use free and low cost Search tools with other platforms, so skip running this if the cost is a concern. 

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing


In [4]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [5]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

In [6]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

In 2026, the most commonly cited AI agent frameworks are **LangGraph**, **CrewAI**, **Microsoft Agent Framework** (the successor path to AutoGen and Semantic Kernel), **OpenAI Agents SDK**, **LlamaIndex Workflows**, **Google ADK**, and **Mastra**. Industry roundups and ecosystem comparisons consistently place LangGraph and CrewAI among the most widely adopted for production agent orchestration, while Microsoft Agent Framework and OpenAI’s SDK stand out as first-party options from major model vendors. ([langchain.com](https://www.langchain.com/resources/ai-agent-frameworks?utm_source=openai))

Popularity depends on how you measure it: GitHub-star rankings and market reports show **AutoGen**, **CrewAI**, **LlamaIndex**, **Semantic Kernel**, and **LangGraph** as major names, with LangGraph often described as the production default for stateful orchestration and CrewAI as a fast path for role-based multi-agent systems. For retrieval-heavy or document-centric agents, LlamaIndex is often highlighted; for Microsoft-heavy stacks, the new Microsoft Agent Framework is increasingly positioned as the unified enterprise choice. ([dreaming.press](https://www.dreaming.press/posts/ai-agent-frameworks-github-ranked-by-stars-2026.html?utm_source=openai))

Checking the traces at https://platform.openai.com/traces

![Search Agent](./assets/images/search-agent.jpg)

We can verify that the Search Agent uses the OpenAI hosted `WebSearchTool` in completing its task.

![Web Search Tool](./assets/images/web-search-tool.jpg)

## 2. The Planner Agent

These are the Pydantic Structured Outputs (`WebSearchItem` and `WebSearchPlan`) used by the planning agent to perform multi-step web research. By passing these schemas to the LLM, you force the model to plan its search strategy explicitly before executing any queries.

#### WebSearchItem (The Individual Query Unit)

- This class represents a single, targeted web search operation.

- `query` (str): The exact search string/keywords the agent intends to send to a search engine (e.g., "most popular AI agent frameworks 2026 ranking adoption").

- `reason` (str): The LLM's internal chain-of-thought rationale explaining why it needs to perform this specific search.

- Why this matters: Forcing the LLM to write a reason improves query quality (via self-reflection) and provides auditability/logging for human developers to see why the agent chose a particular search path.

In [7]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

#### WebSearchPlan (The Orchestrator / Batch Plan)

- This class acts as a container for the complete execution plan.

- `searches` (list[`WebSearchItem`]): A list containing one or more `WebSearchItem` objects.

- Why this matters: Instead of returning a single query or unstructured text, the model returns an array of structured search items. This allows your application code to:

    - Inspect the entire plan upfront.

    - Execute all queries concurrently in parallel using asyncio or thread pools to reduce latency.

    - De-duplicate or filter out unwanted queries before hitting external Search APIs.

In [ ]:
class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

WebSearchPlan.model_json_schema() 

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [9]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='Find recent rankings, comparisons, or analyst lists of leading AI agent frameworks in 2026.', query='most popular AI agent frameworks 2026 ranking'), WebSearchItem(reason='Identify current community adoption signals such as GitHub stars, downloads, and trend reports.', query='AI agent frameworks GitHub stars 2026 LangGraph AutoGen CrewAI Semantic Kernel'), WebSearchItem(reason='Compare major frameworks and their features to assess popularity and market presence.', query='best AI agent frameworks 2026 comparison LangGraph AutoGen CrewAI Semantic Kernel'), WebSearchItem(reason='Look for news, blog posts, and surveys discussing which frameworks are widely used in 2026.', query='2026 AI agent framework survey industry report'), WebSearchItem(reason='Capture newer or emerging frameworks that may have become popular by 2026.', query='emerging AI agent frameworks 2026 popular')])

## 3. The Writer Agent

In [20]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

## 4. The Email Agent

In [10]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [11]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [13]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

## Orchestration by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

In [15]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]

    # * is the unpacking operator.
    # Placing * in front of tasks takes the elements out of the list and 
    # passes them into asyncio.gather() as distinct positional arguments.
    results = await asyncio.gather(*tasks)
    
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [19]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

### Building the pipeline

In [21]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results) 
    await send_report_email(report)  
    print("Hooray!")

Starting research...
Planning searches...
Will perform 5 searches
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!


Looking at the traces: https://platform.openai.com/traces

![Research trace](./assets/images/research-trace.jpg)

From the output of the planner agent, you can review and validate the chain-of-thought reasoning generated. 

```json
{
  "searches": [
    {
      "reason": "Identify current market leaders and widely used frameworks for AI agents in 2026.",
      "query": "most popular AI agent frameworks 2026"
    },
    {
      "reason": "Find recent rankings, surveys, or adoption data comparing agent frameworks.",
      "query": "AI agent framework popularity survey 2026"
    },
    {
      "reason": "Capture up-to-date mentions of major frameworks and their ecosystems.",
      "query": "best AI agent frameworks 2026 LangGraph AutoGen CrewAI Semantic Kernel"
    },
    {
      "reason": "Check developer community signals such as GitHub stars, downloads, and activity.",
      "query": "GitHub stars AI agent frameworks 2026"
    },
    {
      "reason": "Look for recent articles or benchmark-style comparisons discussing framework usage.",
      "query": "AI agent frameworks comparison 2026"
    }
  ]
}
```

![Planner and search agents](./assets/images/planner-and-search-agents.jpg)

The planner agent invoked 5 search agents, running them simultaneously.

The writer agent only starts after all 5 search agents are done with their task.

The email agent uses the `send_email_tool` to send the report generate by the writer agent.

![Writer and email agents](./assets/images/writer-and-email-agent.jpg)